# 03: Demand Forecasting (Phase 3)

This notebook reads the artefacts written by `python main.py forecast` (`src/forecasting.py`). Set `RETRAIN = True` to rerun the full pipeline (about 10 minutes). The full narrative is in `reports/phase3_forecasting_report.md`.

The last section reloads the saved model and shows that it reproduces the CFLP forecast.

In [ ]:
import sys, os, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd().resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, joblib
from IPython.display import Image, display
pd.set_option('display.width', 220, 'display.max_columns', 30, 'display.max_colwidth', 70)
RETRAIN = False
if RETRAIN:
    import main; main.main(['forecast'])
OUT = ROOT / 'data/outputs'
ev = pd.read_csv(OUT / 'forecast_evaluation.csv')
sel = pd.read_csv(OUT / 'selected_forecast_model.csv')
cflp = pd.read_csv(OUT / 'cflp_zone_demand_forecast.csv')
imp = pd.read_csv(OUT / 'feature_importance.csv')

## 1. Stable regime: pre-break test (zone-week, one-step)

In [ ]:
def wape_table(eval_set, mode='one_step', handling='none', gran='W', level='zone_period'):
    x = ev[(ev.granularity==gran)&(ev.eval_set==eval_set)&(ev['mode'].isin([mode,'reference']))&(ev.level==level)&(ev.tier=='all')&(ev.network!='ALL')]
    x = x[(x.handling==handling)|(x.handling=='reference')]
    return x.pivot_table(index=['target','network'], columns='model', values='WAPE').round(3)
wape_table('pre_break_test')

In [ ]:
display(Image('reports/figures/phase3_actual_vs_predicted_network.png'))

## 2. After the October 2017 break

In [ ]:
display(wape_table('post_break_test')); display(wape_table('post_break_test', mode='horizon'))

In [ ]:
x = ev[(ev.granularity=='W')&(ev.eval_set=='post_break_adapted')&(ev.level=='zone_period')&(ev.tier=='all')&(ev.network!='ALL')]
x.pivot_table(index=['target','network'], columns=['model','handling'], values='WAPE').round(3)

In [ ]:
display(Image('reports/figures/phase3_post_break_adaptation.png')); display(Image('reports/figures/phase3_model_wape_comparison_W.png'))

## 3. Zone-demand tiers, granularity, feature importance

In [ ]:
for f in ['phase3_tier_wape_pre_break_test','phase3_weekly_vs_monthly','phase3_feature_importance']:
    display(Image(f'reports/figures/{f}.png'))

## 4. Selected models and the CFLP demand input

In [ ]:
display(sel)
display(cflp.groupby('network')[['forecast_per_period_orders','forecast_per_period_units','historical_avg_per_period_all_orders','historical_avg_per_period_post_break_orders']].sum().round(1))
display(Image('reports/figures/phase3_cflp_demand_forecast_map.png')); display(Image('reports/figures/phase3_zone_total_scatter.png'))

## 5. Reload the saved model and reproduce the forecast

In [ ]:
from src import forecasting as fc
bundle = joblib.load(ROOT / 'models/demand_forecasting_model.pkl')
zd = pd.read_csv('data/processed/zone_daily_demand.csv', parse_dates=['date']); zones = pd.read_csv('data/processed/demand_zones.csv')
panel = fc.build_panel(zd, zones, 'W')
H = 13
hz = fc.horizon(bundle['models'], panel, panel.T - 1, H, actual=False)
fac = fc.recalibration_factors(fc.one_step(bundle['models'], fc.long_frame(panel), np.arange(panel.T - 4, panel.T)))
print(bundle['selection'])
for r in bundle['selection'].itertuples():
    if r.model in fc.ALL_MODELS:
        p = hz[(hz.target==r.target)&(hz.network==r.network)&(hz.model==r.model)].groupby('zone_id').predicted.mean()
        if r.handling == 'recalibrated':
            p = p * fac[(fac.network==r.network)&(fac.target==r.target)&(fac.model==r.model)].factor.iat[0]
        ref = cflp.set_index('zone_id').loc[p.index, f'forecast_per_period_{r.target}']
        print(r.target, r.network, r.model, 'max abs diff vs saved CFLP file:', float(np.abs(p - ref).max()))
    else:
        print(r.target, r.network, r.model, '(regime-reset baseline: computed directly from data; no fitted model needed)')